Install the libraries

In [ ]:
!pip install scikit-learn
!pip install torch

In [ ]:
import numpy as np
import os

# ---------------- CONFIG ----------------
INPUT_FILE = "datasets/clean_eye_data.npz"
OUTPUT_FILE = "datasets/clean_eye_data_lstm.npz"
SEQ_LEN = 5   # number of windows per sequence
# ----------------------------------------


def build_lstm_sequences(X, y, seq_len):
    """
    X: (N, C, T)
    y: (N,)
    Returns:
        X_seq: (N_seq, seq_len, C*T)
        y_seq: (N_seq,)
    """
    N, C, T = X.shape

    X_seq = []
    y_seq = []

    for i in range(N - seq_len + 1):
        seq = X[i:i + seq_len]              # (seq_len, C, T)
        seq_flat = seq.reshape(seq_len, -1) # (seq_len, C*T)

        X_seq.append(seq_flat)
        y_seq.append(y[i + seq_len - 1])    # label of last window

    return np.array(X_seq), np.array(y_seq)


# ---------------- LOAD CLEAN DATA ----------------
data = np.load(INPUT_FILE)

X_clean = data["X"]   # (N, 8, 250)
y_clean = data["y"]   # (N,)

print("Loaded clean data:")
print("X:", X_clean.shape)
print("y:", y_clean.shape)
print("Label distribution:", np.unique(y_clean, return_counts=True))


# ---------------- BUILD SEQUENCES ----------------
X_seq, y_seq = build_lstm_sequences(X_clean, y_clean, SEQ_LEN)

print("\nLSTM-ready data:")
print("X_seq:", X_seq.shape)
print("y_seq:", y_seq.shape)
print("Sequence label distribution:", np.unique(y_seq, return_counts=True))


# ---------------- SAVE ----------------
np.savez(
    OUTPUT_FILE,
    X=X_seq,
    y=y_seq,
    seq_len=SEQ_LEN,
    feature_dim=X_seq.shape[2]
)

print(f"\nSaved LSTM dataset → {OUTPUT_FILE}")

In [ ]:
def temporal_split(X, y, val_ratio=0.2):
    N = len(X)
    split = int(N * (1 - val_ratio))
    return X[:split], y[:split], X[split:], y[split:]


X_train, y_train, X_val, y_val = temporal_split(X_seq, y_seq)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)

Training The Neural Network, Set Up Data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class EEGSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = EEGSequenceDataset(X_train, y_train)
val_ds   = EEGSequenceDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

Model Architecture

In [ ]:
import torch.nn as nn

class EyeStateLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]     
        logits = self.fc(last)
        return logits

Training Loop Log Loss every epoch for train and test set

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = EyeStateLSTM(
    input_dim=X_seq.shape[2],
    hidden_dim=64
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
peak_all_time_test_acc = 0.0

In [ ]:
num_epochs = 100
peak_test_acc = 0.0
peak_test_acc_epoch = 0
record_broken = False

for epoch in range(num_epochs):

    # ---------------- TRAIN ----------------
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)   # move to GPU
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)       # logits (B, 2)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * y_batch.size(0)

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y_batch).sum().item()
        train_total += y_batch.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---------------- TEST ----------------
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)   # move to GPU
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            test_loss += loss.item() * y_batch.size(0)

            preds = torch.argmax(outputs, dim=1)
            test_correct += (preds == y_batch).sum().item()
            test_total += y_batch.size(0)

    test_loss /= test_total
    test_acc = test_correct / test_total

    # ---------------- CHECKPOINTS ----------------
    if test_acc > peak_test_acc:
        peak_test_acc = test_acc
        peak_test_acc_epoch = epoch + 1

    if test_acc > peak_all_time_test_acc:
        peak_all_time_test_acc = test_acc
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch + 1,
            "test_acc": test_acc
        }, "assets/eye_model.pth")

        print(
            f" New BEST model saved | "
            f"Test Acc: {test_acc*100:.2f}% | Epoch {epoch+1}"
        )
        record_broken = True

    # ---------------- LOG ----------------
    print(
        f"Epoch {epoch+1:03d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}% | "
        f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc*100:.2f}%"
    )

print(f"\nPeak Test Accuracy: {peak_test_acc*100:.2f}%")
print(f"Peak Accuracy Epoch: {peak_test_acc_epoch}")

if record_broken:
    print(
        f" All-time best test accuracy: "
        f"{peak_all_time_test_acc*100:.2f}% "
        f"(epoch {peak_test_acc_epoch})")

Metrics

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)    # move to GPU
        y_batch = y_batch.to(device)

        outputs = model(X_batch)        # logits (B, 2)
        preds = torch.argmax(outputs, dim=1)

        all_preds.append(preds.cpu().numpy())
        all_labels.append(y_batch.cpu().numpy())

# Concatenate batches
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Eyes Open", "Eyes Closed"]
    )
)